---
title: Planning and preferences
author:
  - name: Alex Lepauvre
    orcid: 0000-0002-4191-1578
    corresponding: true
    email: alex_francois.lepauvre@tu-dresden.de
    roles: []
    affiliations:
      - Chair of Cognitive Computational Neuroscience, Faculty of Psychology, TUD Dresden University of Technology, Dresden 01187, Germany
  - name: Stefan Kiebel
    orcid:  0000-0002-5052-1117
    corresponding: true
    roles: []
    affiliations:
      - Chair of Cognitive Computational Neuroscience, Faculty of Psychology, TUD Dresden University of Technology, Dresden 01187, Germany
keywords:
  - Decision making
  - Reinforcement learning
abstract: |
  Human decision making is often described in terms of value-based planning, yet behavior in complex tasks systematically deviates from optimal predictions. These deviations are typically captured using flexible bias parameters, leaving their underlying structure unclear.
  Here, we examine behavior in a complex sequential decision task and test whether such deviations can be explained in a more principled way. We show that behavior can be well described by a combination of value-based planning and a small set of simple, state-dependent preference terms motivated by task structure. These preferences reflect coarse properties of the environment, including offer characteristics, action costs, and transition structure, and substantially improve model fit over planning alone. Importantly, they account for a meaningful portion of the variability captured by more flexible bias models.
  Across analyses, we find that preferences alone are insufficient to explain behavior, but jointly contribute with planning to guide decisions. Trial-level analyses further show that deviations from preference-consistent behavior arise when preference signals are weak or opposed by strong decision values. In addition, trials with stronger preference signals are associated with faster response times, consistent with reduced reliance on computationally demanding planning.
  Together, these results suggest that deviations from planning are not purely arbitrary, but reflect structured, task-informed preferences that complement value-based evaluation. This provides a simple and interpretable account of behavior in complex decision problems. 
date: last-modified
number-sections: true
bibliography: references.bib
---

## Introduction

The human brain is remarkable in its capacity to learn and solve complex tasks efficiently. Even performing the mundane tasks from our daily life (planning the day, cooking a meal) constitute complex planning problems requiring significant computational resources to resolve somewhat adequately. And while in recent years, artificial intelligence have been able to surpass us for certain tasks (playing chess...), they systematically require more resources to do so (more training to learn to solve a problem, as well as more energy required at each problem solving iteration). 

This extraordinary capacity is typically studied by applying reinforcement learning algorithms, where decision making problems are operationalized using Markov Decision Problems (MDP) modelling the environment in terms of states and rewards associated with decision performed by participants in each states. Many algorithms have been developped to identify the values associated with each action in each state to determine the optimal decision participants have to perform in each state to maximize their reward over time (rather than in a single state, CITE SUTTON AND BARTO). While these algoritms have been instrumental in understanding key aspects of human decision making (HERE CITE SOME REVIEW), they typically fall short in two regards. In complex tasks, humans behaviour deviates systematically from optimal predictions from these models. They are not biologically plausible as the computation complexity grows exponentially with the number of states involved, making them biologically intractable for any real life problem.

Both these limitations are thought to be intertwined, as it is generally assumed that humans behave suboptimally in complex tasks because they do not have the computational resources to compute the optimal solution. Instead, they have to identify the best possible solution given limited resources, an idea known as bounded rationality, which is reflect in participants behaviour. Practically, deviations from optimality are capture using flexible bias parameters to account for the systematic deviations from optimality. These bias are often treated as necessary to reveal the underlying planning more clearly. The structure of these preferences is rarely investigated and therefore remains poorly understood.

In this paper, we examine human behaviour in a large state space (448 states) to investigate whether deviations from optimality reflect the structure of the task. 

In [1]:
# Import all packages

# General utilities:
import os 
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import warnings

# Stats
import pymc as pm
import arviz as az
import bambi as bmb
from scipy.special import logit

# Custom packages:
from stabst.MarkovDecisionProcess import MDP
from stabst.TaskConfig import LimitedEnergyTask
from stabst.utils import avg_reduce_mdp, abstract2ground_value

# Set random seed for reproducibility:
np.random.seed(42)
warnings.filterwarnings("ignore")

n_samples = 1000
n_warmup = 1000
n_chains = 4

In [2]:
# Download the data if needed:
if not os.path.exists('./data/raw_data/all_participants_data.csv'):
    if not os.path.exists('./data/raw_data'):
        os.makedirs('./data/raw_data')
    url = 'https://raw.githubusercontent.com/fmott/context_dependent_planning/4d239b721749adabb8fe8f1d8ac2d1ecdeba17cf/data/behaviour/data_all_participants_20220215120148.csv'
    os.system(f'wget {url} -O ./data/raw_data/all_participants_data.csv')
if not os.path.exists('./data/raw_data/all_participants_age_gender.csv'):
    if not os.path.exists('./data/raw_data'):
        os.makedirs('./data/raw_data')
    url = 'https://raw.githubusercontent.com/fmott/context_dependent_planning/4d239b721749adabb8fe8f1d8ac2d1ecdeba17cf/data/behaviour/age_gender.csv'
    os.system(f'wget {url} -O ./data/raw_data/all_participants_age_gender.csv')
# Load the data:
beh_data = pd.read_csv('./data/raw_data/all_participants_data.csv')
demographic_data = pd.read_csv('./data/raw_data/all_participants_age_gender.csv', sep=";")

# Extract demographic information:
n_participants = demographic_data.shape[0]
n_female = demographic_data['gender (m = 1, f = 2)'].value_counts()[2]
mean_age = demographic_data['age'].mean()
std_age = demographic_data['age'].std()

# ===================================================================
# Data preprocessing:
# Remove nans:
beh_data = beh_data.dropna()
# Remove timeout:
beh_data = beh_data[beh_data["timeout"] == 0]
# Flip responses: 1 = accept:
beh_data["response"] = (beh_data["response"] == 0).astype(int)
# Make trial 1 based
beh_data["trial"] = beh_data["trial"] + 1
# Generate future cost based on the transitions:
transitions_costs = {
    0: [1, 1],
    1: [2, 1],
    2: [1, 2],
    3: [2, 2]
}
beh_data["fc"] = [transitions_costs[row["transition"]][1] for _, row in beh_data.iterrows()]


## Methods

### Participants and experimental design
For this paper, we reused data from a previously published study in whichy`{python} f"{n_participants:.0f}"` participants (`{python} f"{n_female:.0f}, mean age={mean_age:.1f}, SD={std_age:.1f}"`) took part in a sequential decision making task (@ott2022forward, @ott2022forward-data). The study was approved by the Institutional Review Board of the Technische Universität Dresden and conducted in accordance to ethical standards of the Declaration of Helsinki.

In the task, participants were instructed to gather as many points as possible throughout the task. In each trial, participants were presented with offers associated with a reward of 1, 2, 3 or 4 points which they could either accept or reject. To accept an offer, participants have to pay an energy cost of either 1 or 2 energy points. In case they reject the offer, they gain 1 energy point. Participants energy level was capped, there energy can therefore range from 0 (depleted energy) to 6. Offers vary in each trial following a uniform distribution (equal probability of each offer in each trial). Costs remain fixed for 4 trials segments and participants were informed of the price of the current 4 trials and future 4 trials. The costs of the segment beyond the current horizon was randomly picked such that each costs transitions (from current low cost to future high and low cost and vice versa) were equally probable throughout the experiment. If participants accept an offer that they cannot afford (current energy level below current cost), no reward was awarded. 

![Experimental design (from @ott2022forward): (A) Single trial procedure, each frame represents a step within the trial with duration of each step above. Top row depicts participants accepting, bottom row depicts participants rejecting. In each trial, participants are presented with N golden cups, the number of cups symbolizes the amount of reward. The blue bar represents energy level, they yellow bar represents accumulated reward so far, lightning symbols at the bottom right of each frame represent current cost (left) and future cost (right). (B) Between trial costs dependencies. A segment consists of 4 trials within which cost is fixed, and participants are aware of the cost in the current and future 4 trials but not beyond. (C) Transition structure. Cost can transition from low cost (LC) to high cost (HC), LC to LC, HC to LC and HC to HC](images/experimental_design.jpg){#fig-design fig-alt="Experimental design"}

Accordingly, participants have to decide whether to accept or reject an offer based on the current offer and cost ratio. They also take into account the impact of their current decision on their future capabilities to accept high offers, as systematically accepting offer will deplete energy fast, hindering their capacity to collect more reward. Therefore, maximizing overall return is non trivial in this task and requires complex planning. 


### Optimal planning model of choice behaviour
To model optimal behaviour in the task, we operationalized our task as a Markov Decision Process with the following tuples:

$$
MDP = (S, A, P, R)
$$

Where $A$ is the set of possible actions in our task ($accept=1, reject=0$), $P$ is the transitional probability from a given state to all other states ($P=P_{\forall s \in S}(s'|s, a)$), and $R$ is the reward function characterizing the reward participants can obtain in a given state for each action ($R=R_{\forall s \in S}(s|a)$). $S$ represent all possible states in our task, where each state $s$ consists of a combination of the experimental variables in our task:  $S=E \times O \times CC \times FC \times T$, with energy $E={0, 1, ..., 6}$, offer $O={1, 2, 3, 4}$, current segment cost $CC={1, 2}$, future segment cost $FC={1, 2}$ and trial $T={1, 2,..., 13}$. Time steps (i.e. trials) has to be incorporated in the state space to account for the cost being trial dependent. Furthermore, despite knowledge horizon going only until trial 8, we incorporated one segment beyond the known horizon to ensure that future beyond known horizon is considered, otherwise the optimal solution would seek to reach minimal energy level by the end of the 2 segments, which would be detrimental to long term planning.

By extending the time horizon to one trial beyond what is known, we can operationalize the task as an episodic MDP and solve it for the finite horizon of 13 trials. For the trials beyond the known horizon the uncertainty associated with the offer is compounded by the uncertainty related to the costs transition. In trials in the known horizon, the probability of the next state reflects the offer related stochasticity while the result is deterministic, based on participants actions and structural constraints of our task (see @ott2022forward for more details and [here](LINK)). 

We applied backward induction to the MDP to compute value associated with each state and the value associated with each action in each state (V and Q functions respectively, see @sutton1998reinforcement). If participants behave optimally, their behaviour should reflect a softmax function over the value associated with each action in a given state, which in the case of a binary decision problem simplifies to the logit function (see @lepauvre2026preferences-code for proof):

$$
\begin{align}
P(a=1) &= \frac{1}{1+e^{DV}} \\
where\ DV &= Q(s, a=1) - Q(s, a=0)
\end{align}
$$


In [3]:
# ===================================================================
# Task MDP:
# Create the task and its parameters (transition probability, reward...):
task = LimitedEnergyTask(O=[1, 2, 3, 4], p_offer=[1/4] * 4)
task.build()

# Create full MDP and compute solution for later reference:
gamma = 1
task_mdp = MDP(task.states, task.tp, task.r, gamma, s2i=task.s2i)
V_full, Q_full = task_mdp.backward_induction()

# Add decision values to the data frame:
dv = Q_full[:, 1] - Q_full[:, 0]
# Loop through each trial to set DV:
dv_trials = []
for trial_i, trial in beh_data.iterrows():
    e, o, cc, t = trial.energy, trial.reward, trial.energy_cost, trial.trial
    fc = transitions_costs[trial.transition][1]
    dv_trials.append(dv[task.s2i[(e, o, cc, fc, t)]])
beh_data['dv'] = dv_trials
# Compute offer specific decision value regressors:
beh_data['dv_23'] = beh_data['dv'].to_numpy() * (beh_data['is_2'].to_numpy() + beh_data['is_3'].to_numpy())
beh_data['dv_14'] = beh_data['dv'].to_numpy() * (beh_data['is_1'].to_numpy() + beh_data['is_4'].to_numpy())


### Modelling participants preferences
In addition from the planning component, it was previously observed that participants behaviour differs significantly depending on the offer. Specifically, it was observed that participants respond significantly faster yet less optimally when offers were either high or low (o=1 and o=4) compared to intermediate offers (o=2 and o=4). This was interpreted as indication that participants consider that the task constitutes of two separate offer based context, and inferred that the intermediate offer context required more careful planning than the extreme offer context. 

In this work, we aimed to investigate whether instead of a binary contextualization of the task, participants behaviour reflects a weighted combination of a planning component with preferences reflecting the structure of the task. Specifically, we hypothesized that the data can be explained by a weighted combination of preferences associated with each levels of each factors of the state space. This idea can be formalized using the following model:

$$
\begin{align*}
P(a=1) &= \frac{1}{1+e^{\eta}} \\
where\ \eta &= \beta_{plan} \times Q(s, a=1) - Q(s, a=0) + \mathbf{X_{pref}}\mathbf{\beta}
\end{align*}
$$

Where $\mathbf{X_{pref}}$ is a $[M \times N]$ (m=measurements, N=experimental levels) matrix dummy coding each factors of the experimental design (energy, offer, current and future costs), and the $\mathbf{\beta}$ is a vector of weights estimated for each levels of each factors of the experimental design. In other words, in addition to the planning component, we estimated weights associated with each experimental factors to compute a compound preference score. Beyond the mere combination of planning and preferences, we sought to investigate whether participants behaviour reflects an interaction between the two, whereby if participants rely more or less on planning depending on whether they have weak or strong preferences in a particular state (preferences scores closer or further away from 0). This required us to estimate the latent preference score component from the data and test for possible interaction with the planning component. However, as our interaction relate not to the raw preference scores but to their strength, we transformed the preference score to entropy. In other words:

$$
\begin{align*}
P(a=1) &= \frac{1}{1+e^{\eta}} \\
where\ \eta_{preference} &= \beta_{plan} \times DV + Preference + \beta_{Interaction}(H(Preference) \times DV) \\
with\ Preference &= \mathbf{X_{pref}}\mathbf{\beta_{pref}}
\end{align*}
$$

In addition to this model, we fitted both the planning model and hybrid model from Ott and colleagues (@ott2022forward). Specifically:

$$
\begin{align*}
\eta_{planning} =& \beta{plan} \times DV + \\
&\beta_{basic}\mathbf{I}_{basic} + \beta_{maxE}\mathbf{I}_{maxE} + \\
&\beta_{minE_{LC}}\mathbf{I}_{minE_{LC}} + \beta_{minE_{HC}}\mathbf{I}_{minE_{HC}} 
\end{align*}
$$


$$
\begin{align*}
\eta_{hybrid} =& \beta{plan23} \times DV_{23} + \beta{plan14} \times DV_{14} + \\
&\beta_{O1}\mathbf{I}_{O1} + \beta_{O2}\mathbf{I}_{O2} + \beta_{O3}\mathbf{I}_{O3} + \beta_{O4}\mathbf{I}_{O4} + \\
&\beta_{basic}\mathbf{I}_{basic} + \beta_{maxE}\mathbf{I}_{maxE} + \\
&\beta_{minE_{LC}}\mathbf{I}_{minE_{LC}} + \beta_{minE_{HC}}\mathbf{I}_{minE_{HC}} 
\end{align*}
$$

Where $\mathbf{I}$ indicate dummy regressor encoding specific experimental conditions: $\mathbf{I}_{basic}$ encode trials where energy is sufficient to accept the offer and inferior to 6, $\mathbf{I}_{maxE}$ encodes trials where energy is equal to 6, $\mathbf{I}_{minE_{LC}}$ for trials where energy is too low to accept the offer when cost is low,  $\mathbf{I}_{minE_{HC}}$ for trials where energy is too low to accept the offer when cost is high, $\mathbf{I}_{minE_{O1-4}}$ for trials with offer 1-4 where energy is sufficient to accept. 


In [4]:
# Choice behaviour model with preferences and decision values:
def preference_model(
        y: np.array,
        decision_values: np.array,
        pref_regressors: pd.DataFrame,
        subject_index: np.array,
        subject_labels: np.array      
):
    '''
    Parameters
    ----------
    y : np.array [N samples, ]
        Observed binary data (1, 0...)
    decision_values : np.array [N samples, ]
        Decision values to regress onto the observed data
    pref_regressors : np.array [N samples, M regressors]
        Regressor to fit participants preference for. We can have M regressors
    subject_index : np.array  [N samples, ]
        Index of the subject associated with each observation
    subject_labels : np.array  [N subjects, ]
        Single identifier of each subject
    coords : dict  
        "subject": subj_labels, 
        "coef": ["intercept", "slope"],
        The subject maps the data to each subject, the coef are for the coefficients
    b_prior_mean : Optional[float], optional
        Prior mean of each beta parameters, by default 0
    b_prior_sigma : Optional[float], optional
        Prior variance of the population level distribution of the beta, by default 2
    s_prior_sigma : Optional[float], optional
        Prior between subjects variance, by default 2
    n_drawss : Optional[int], optional
        Number of draws for the posterior, by default 1000
    n_tuning_draws : Optional[int], optional
        Number of tuning draws, by default 1000
    Returns
    -------
    tuple[pm.Model, arviz.InferenceData]
        pm.model : pymc model object
        idata : arviz inference data
    """
    '''
    # Get dimensions:
    n_obs = y.shape[0]
    n_groups = subject_labels.shape[0]
    n_pref = pref_regressors.shape[1]

    # Create intercept:
    intercept = np.ones(n_obs)

    # Set coordinates:
    coords = {
        "subject": subject_labels,
        "coef_intercept": ["B_intercept"],
        "coef_planning": ["B_plan"],
        "coef_pref": ["B_" + col for col in pref_regressors.columns],
        "coef_interaction": ["slope"],
    }


    # Model:
    with pm.Model(coords=coords) as planning_preferences_interaction_model:
        # Data:
        y_obs = pm.Data("y_obs", y)
        intercept = pm.Data("intercept", intercept)
        planning = pm.Data("planning", decision_values)
        preferences = pm.Data("preferences", pref_regressors)
        subj_idx = pm.Data("subj_idx", subject_index.astype("int32"))

        # Hyperpriors:
        # Intercept term
        beta_intercept = pm.Normal("beta_intercept", mu=0, sigma=2, dims="coef_intercept")
        sigma_intercept = pm.HalfNormal("sigma_intercept", sigma=2, dims="coef_intercept")
        # Planning term:
        beta_planning = pm.Normal("beta_planning", mu=0, sigma=2, dims="coef_planning")
        sigma_planning = pm.HalfNormal("sigma_planning", sigma=2, dims="coef_planning")
        # Preference terms:
        beta_pref = pm.Normal("beta_pref", mu=0, sigma=2, dims="coef_pref")
        sigma_pref = pm.HalfNormal("sigma_pref", sigma=2, dims="coef_pref")
        # Interaction term:
        beta_interaction = pm.Normal("beta_interaction", mu=0, sigma=2, dims="coef_interaction")
        sigma_interaction = pm.HalfNormal("sigma_interaction", sigma=2, dims="coef_interaction")

        # Offset parameters:
        z_intercept = pm.Normal("z_intercept", 0, 1, dims=("subject", "coef_intercept"))
        z_planning = pm.Normal("z_planning", 0, 1, dims=("subject", "coef_planning"))
        z_biases = pm.Normal("z_biases", 0, 1, dims=("subject", "coef_pref"))
        z_interaction = pm.Normal("z_interaction", 0, 1, dims=("subject", "coef_interaction"))

        # Centered parameters:
        beta_intercept_sub = pm.Deterministic("beta_intercept_sub", beta_intercept + z_intercept * sigma_intercept, 
                                              dims=("subject", "coef_intercept"))
        beta_planning_sub = pm.Deterministic("beta_planning_sub", beta_planning + z_planning * sigma_planning, 
                                             dims=("subject", "coef_planning"))
        beta_pref_sub = pm.Deterministic("beta_pref_sub", beta_pref + z_biases * sigma_pref, 
                                         dims=("subject", "coef_pref"))
        beta_interaction_sub = pm.Deterministic("beta_interaction_sub", beta_interaction + z_interaction * sigma_interaction, 
                                                dims=("subject", "coef_interaction"))
        
        # Estimate the score of the bias (i.e. weighted sum of each of the biases regressors):
        preference = pm.Deterministic('preference', (beta_pref_sub[subj_idx] * preferences).sum(axis=-1))
        
        # Convert the bias back onto probability space:
        pi_prior = pm.Deterministic("pi_prior", pm.math.sigmoid(preference))

        # Compute the entropy:
        entropy = pm.Deterministic("entropy", -pi_prior * pm.math.log(pi_prior) - (1-pi_prior) * pm.math.log(1 - pi_prior))
        
        # Eta parameter is the weighted sum of the intercept, the bias, the planning values and 
        # the interaction between the entropy of the bias and the planning
        eta = (
            beta_intercept_sub[subj_idx, 0] * intercept
            + preference
            + beta_planning_sub[subj_idx, 0] * planning
            + beta_interaction_sub[subj_idx, 0] * (entropy * planning)
        )
        
        # Expected values:
        p = pm.Deterministic("p", pm.math.sigmoid(eta))

        # Likelihood 
        pm.Bernoulli("y", p=p, observed=y_obs)

        # Sampling:
        idata = pm.sample(
            draws=n_samples,
            tune=n_warmup,
            chains=n_chains,
            target_accept=0.85,
            idata_kwargs={"log_likelihood": True},
        )

    return idata

In [5]:
# Preparing regressors:
# Compute categorical regressors that should be somewhat similar to our priors:
# Categorical offer regressor for high and low offer
beh_data['is_12'] = beh_data['is_1'].to_numpy() + beh_data['is_2'].to_numpy()
beh_data['is_34'] = beh_data['is_3'].to_numpy() + beh_data['is_4'].to_numpy()
# Continuous regressor for high and low offer:
beh_data['high_vs_low'] = beh_data['is_34'] - beh_data['is_12']

# Categorical costs regressor
beh_data['is_lc'] = (beh_data['energy_cost'] == 1).astype(int).to_numpy()
beh_data['is_hc'] = (beh_data['energy_cost'] == 2).astype(int).to_numpy()
# Categorical future costs regressor
beh_data['is_lfc'] = (beh_data['fc'] == 1).astype(int).to_numpy()
beh_data['is_hfc'] = (beh_data['fc'] == 2).astype(int).to_numpy()

# Categorical transition regressor
beh_data['is_trans1'] = (beh_data['transition'] == 0).to_numpy()
beh_data['is_trans2'] = (beh_data['transition'] == 1).to_numpy()
beh_data['is_trans3'] = (beh_data['transition'] == 2).to_numpy()
beh_data['is_trans4'] = (beh_data['transition'] == 3).to_numpy()

# Categorical energy regressor:
beh_data['e_is_0'] = (beh_data['energy'] == 0).to_numpy()
beh_data['e_is_1'] = (beh_data['energy'] == 1).to_numpy()
beh_data['e_is_2'] = (beh_data['energy'] == 2).to_numpy()
beh_data['e_is_3'] = (beh_data['energy'] == 3).to_numpy()
beh_data['e_is_4'] = (beh_data['energy'] == 4).to_numpy()
beh_data['e_is_5'] = (beh_data['energy'] == 5).to_numpy()
beh_data['e_is_6'] = (beh_data['energy'] == 6).to_numpy()

# Random effects
subj_idx_raw, subj_labels = pd.factorize(beh_data["vpn"])




### Model fitting and comparison

The model described above were fitted on participants responses data as Bayesian hierarchical logistic regressions using PYMC (@abril2023pymc) and Bambi (@Capretto2022), (see @lepauvre2026preferences-code for the exact models). We excluded trials where participants reaction time (RT) exceeded the response window (5s) or didn't provide a response. Across models, the following parameters were left to vary across participants (i.e. random slope): $\beta_{plan},\ \beta_{plan23},\ \beta_{plan14},\ \beta_{pref},\ \beta_{Interaction},\ \beta_{basic},\ \beta_{O1},\ \beta_{O1},\ \beta_{O2},\ \beta_{O3},\ \beta_{O4}$, while the rest were fixed across participants. For all parameters, we used weakly informative hyperprior distributions $\mu \sim \mathcal{N}(0, 2)$ and $\sigma \sim Halfnormal(0, 2)$. All models were fitted using 4 chains of 8000 sample each (4000 warmups), resulting in 16000 samples in total. 

We used Pareto-smoothed importance sampling to approximate leave one out cross validation (PSIS-LOO) to estimate the expected log pointwise predictive density (elpd) which we used to compare the fit of these different models. 

In [6]:
# Fitting the models:
traces = {}
# ===================================================================
# Preference model:
if os.path.exists("./data/bids/limited_energy/derivatives/models/preferences_model_trace.nc"):
    idata = az.from_netcdf("./data/bids/limited_energy/derivatives/models/preferences_model_trace.nc")
    traces['preferences_model'] = idata
else:
    preference_columns = [
        'is_1', 'is_2', 'is_3', 'is_4', 
        'is_lc', 'is_hc', 
        'is_trans1', 'is_trans2', 'is_trans3', 'is_trans4', 
        'e_is_0', 'e_is_1', 'e_is_2', 'e_is_3', 'e_is_4', 'e_is_5', 'e_is_6'
    ]
    idata = preference_model(beh_data['response'],  # Subjects responses
                             np.squeeze(beh_data[['dv']].to_numpy()),  # Optimal decision values
                             beh_data[preference_columns],  # Preferences regressors
                             subj_idx_raw, subj_labels)

    # Add the idata to the rest:
    traces['preferences_model'] = idata
    # Save the trace to file:
    if not os.path.exists("./data/bids/limited_energy/derivatives/models/"):
        os.makedirs("./data/bids/limited_energy/derivatives/models/")
    az.to_netcdf(idata, "./data/bids/limited_energy/derivatives/models/preferences_model_trace.nc")

In [7]:
# ===================================================================
# Hybrid model from Ott's
if os.path.exists("./data/bids/limited_energy/derivatives/models/hybrid_model_trace.nc"):
    idata = az.from_netcdf("./data/bids/limited_energy/derivatives/models/hybrid_model_trace.nc")
    traces['hybrid_model'] = idata
else:
    # Define the priors:
    priors = {
        'dv_23': bmb.Prior('Normal', mu=0, sigma=2),
        'dv_14': bmb.Prior('Normal', mu=0, sigma=2),
        'is_basic_1': bmb.Prior('Normal', mu=0, sigma=2),
        'is_basic_2': bmb.Prior('Normal', mu=0, sigma=2),
        'is_basic_3': bmb.Prior('Normal', mu=0, sigma=2),
        'is_basic_4': bmb.Prior('Normal', mu=0, sigma=2),
        'is_full_energy': bmb.Prior('Normal', mu=0, sigma=2),
        'is_low_energy_LC': bmb.Prior('Normal', mu=0, sigma=2),
        'is_low_energy_HC': bmb.Prior('Normal', mu=0, sigma=2),
        'dv_23|vpn': bmb.Prior('Normal', mu=0, sigma=bmb.Prior('HalfNormal', sigma=2)),
        'dv_14|vpn': bmb.Prior('Normal', mu=0, sigma=bmb.Prior('HalfNormal', sigma=2)),
        'is_basic_1|vpn': bmb.Prior('Normal', mu=0, sigma=bmb.Prior('HalfNormal', sigma=2)),
        'is_basic_2|vpn': bmb.Prior('Normal', mu=0, sigma=bmb.Prior('HalfNormal', sigma=2)),
        'is_basic_3|vpn': bmb.Prior('Normal', mu=0, sigma=bmb.Prior('HalfNormal', sigma=2)),
        'is_basic_4|vpn': bmb.Prior('Normal', mu=0, sigma=bmb.Prior('HalfNormal', sigma=2)),
    }

    hybrid_model = bmb.Model(
        "response ~ dv_23 + dv_14 + is_basic_1 + is_basic_2 + is_basic_3 + is_basic_4 + is_full_energy + is_low_energy_LC + is_low_energy_HC + "
        " + (dv_23 + dv_14 + is_basic_1 + is_basic_2 + is_basic_3 + is_basic_4|vpn)",
            beh_data, 
            family="bernoulli"
        )
    # Add the idata to the rest:
    traces['hybrid_model'] = hybrid_model.fit(
        draws=n_samples, tune=n_warmup, chains=n_chains, target_accept=0.85, idata_kwargs={"log_likelihood": True}
    )
    # Get the predicted responses:
    hybrid_model.predict(traces['hybrid_model'], kind="response_params", inplace=True)

    # Save the trace to file:
    az.to_netcdf(traces['hybrid_model'], "./data/bids/limited_energy/derivatives/models/hybrid_model_trace.nc")

In [8]:
# ===================================================================
# Planning model from Ott's
if os.path.exists("./data/bids/limited_energy/derivatives/models/planning_model_trace.nc"):
    idata = az.from_netcdf("./data/bids/limited_energy/derivatives/models/planning_model_trace.nc")
    traces['planning_model'] = idata
else:
    # Define the priors:
    priors = {
        'dv_23': bmb.Prior('Normal', mu=0, sigma=2),
        'dv_14': bmb.Prior('Normal', mu=0, sigma=2),
        'is_basic_1': bmb.Prior('Normal', mu=0, sigma=2),
        'is_basic_2': bmb.Prior('Normal', mu=0, sigma=2),
        'is_basic_3': bmb.Prior('Normal', mu=0, sigma=2),
        'is_basic_4': bmb.Prior('Normal', mu=0, sigma=2),
        'is_full_energy': bmb.Prior('Normal', mu=0, sigma=2),
        'is_low_energy_LC': bmb.Prior('Normal', mu=0, sigma=2),
        'is_low_energy_HC': bmb.Prior('Normal', mu=0, sigma=2),
        'dv_23|vpn': bmb.Prior('Normal', mu=0, sigma=bmb.Prior('HalfNormal', sigma=2)),
        'dv_14|vpn': bmb.Prior('Normal', mu=0, sigma=bmb.Prior('HalfNormal', sigma=2)),
        'is_basic_1|vpn': bmb.Prior('Normal', mu=0, sigma=bmb.Prior('HalfNormal', sigma=2)),
        'is_basic_2|vpn': bmb.Prior('Normal', mu=0, sigma=bmb.Prior('HalfNormal', sigma=2)),
        'is_basic_3|vpn': bmb.Prior('Normal', mu=0, sigma=bmb.Prior('HalfNormal', sigma=2)),
        'is_basic_4|vpn': bmb.Prior('Normal', mu=0, sigma=bmb.Prior('HalfNormal', sigma=2)),
    }

    planning_model = bmb.Model(
        "response ~ dv + is_basic + is_full_energy + is_low_energy_LC + is_low_energy_HC + "
        " + (dv + is_basic|vpn)",
            beh_data, 
            family="bernoulli"
        )
    # Add the idata to the rest:
    traces['planning_model'] = planning_model.fit(
        draws=n_samples, tune=n_warmup, chains=n_chains, target_accept=0.85, idata_kwargs={"log_likelihood": True}
    )
    # Predict in place for the bambi models, done by default in pymc:
    planning_model.predict(traces['planning_model'], kind="response_params", inplace=True)
    az.to_netcdf(traces['planning_model'], "./data/bids/limited_energy/derivatives/models/planning_model_trace.nc")

In [9]:
model_comparison = az.compare(traces)

# Compute the LOO for each model and participants:
loo_results = {}
for model_name, idata in traces.items():
    loo_results[model_name] = az.loo(idata, pointwise=True)
loos_df = pd.DataFrame({
    "vpn": beh_data['vpn'],
    **{model_name: loo_results[model_name].loo_i for model_name in loo_results.keys()}
})
# Sum within subject:
loos_sub = loos_df.groupby('vpn').sum().reset_index()

In [10]:
# Add the predicted values to the data frame:
beh_data["P(a=1)_preference"] = np.mean(traces['preferences_model'].posterior["p"], axis=(0, 1))
beh_data["P(a=1)_hybrid"] = np.mean(traces['hybrid_model'].posterior["p"], axis=(0, 1))
beh_data["P(a=1)_planning"] = np.mean(traces['planning_model'].posterior["p"], axis=(0, 1))

Furthermore, we perform model comparison within each single subject to estimate the robustness of the winning model in our population sample, but computing the sum of the pointwise predictive accuracy for each participant and model, yielding a score for each participant and model. We further predicted that participants for whom the preferences model fits the best, we should observed a positive correlation between preferences entropy and total return, as participants with stronger preferences should deviate most from optimal decision making reflected in the decision values components. To test this prediction, we tested for a correlation between average preferences entropy and total score. 

### Reaction times analysis
Following the observation that modelling choice behaviour as a function of both preferences and decisions as well as the interaction between the two outperforms models assuming that different offers are treated as separate contexts, we hypothesized that the degree to which participants engage in effortful foward planning in a given state depends on the strength of their preferences. If that is the case, this should be reflected in participants reaction time (RT), such that the stronger their preferences (i.e. the lower their preferences entropy), the lower their RT. Importantly, as it was previously observed that participants RT increases when decision values are close to 0, which was interepreted as evidence that participants are able to reach a decision quicker when the value asssociated with one decision is much higher than that associated with any other decisions @ott2022forward. In this paper, they further observed that participant RT depended more on decision values in intermediate offers (2 and 3) compared to extreme ones (1 and 4), which they interpreted as evidence that participants consider these offer groups as separate contexts, with the latter requiring more forward planning that the former. 

Here, we predicted that the driving factors of reaction time was not simply offer groups but rather the strength of participants state-specific preferences. In other words, it is possible that the reason participants treat intermediate and extreme offers differently is because of different overall preferences associated with states presenting such offer. By fitting preferences directly in the previous model we can investigate with a greater granularity how it shapes reaction time combined with decision values. We modelled the log of reaction time using the following model:

$$
log(RT) = \beta_{plan} * DV + \beta_{pref} * H(preferences) + \beta_{interaction} * (DV : H(preferences))
$$

Where $preferences$ is the preference score fitted from the preference choice model above, and $H$ stands for the binomial entropy function. We compared this model to @ott2022forward RT model to test whether the use of preference fits the data better than a dichotomous distinction between intermediate and extreme offers, which would indicate that participants consider the strength of their preferences to determine how much they should invest in planning and that the observation of the distinction between intermediate and extreme offers is a byproduct thereof. 

### Task structure importance
Our choice and reaction time results replicate and extend on @ott2022forward findings. We indeed observed that deviation from optimal value based decisions depends strongly on offer as well as on energy. Our results further indicates that these deviations reflect composite preferences derived from the structure of the task. Our results also suggest that not all dimensions of the task are weighted equally in participants preferences. To further quantify the importance associated with each factor of the task (offer, energy, current and future cost), we performed a variance partitioning analysis by computing the coefficient of determiniation associated with each factor @tonidandel2010determining:

$$
R_O^2 = 1 - \frac{\sum{(y-\hat{y})^2}}{\sum{(y-\bar{y})^2}}
$$

Where $\hat{y}$ are the responses fitted by the model without the factor in question and $\bar{y}$ are the responses fitted by the full model. The $R_O^2$ therefore quantifies the proportion of responses that are accounted for by each factors in the model.

A final question we sought to investigate is whether this hierarchy in terms of factors on which preferences are based is arbitrary or instead reflects optimality 

## Results

### Choice behaviour
In line with @ott2022forward, we observe that a model taking only planning component into account does not perform as well compared to taking more flexible biases into account. Importantly, the preference model was found to significantly outperform the hybrid model, indicating that participants biases do reflect the structure of the task (see @fig-2 A). When investigating the fitted preferences parameters, we observe that participants exhibit strong offer based preferences, such that they tend to reject low offers (1 and 2) more than dicated by the decision values and accept larger offers more than they should (see @fig-2 B), and this bias is more extreme for more extreme offers (1 and 4) than intermediate ones (2 and 3), replicating Ott's impact of offers on decision making. For costs, the estimated preference parameters all significantly overlap 0, indicating that participants do not show any bias away from optimal decision based on task. In terms of energy, participant show strong preferences when energy is at 0 (systematically reject) and at 6 (almost systematically accept regardless of any other conditions, see below). In intermediate values, participants show a midly incremental preference, such that they have a tendency to reject offers slightly more than they should when energy is low, but are increasingly likely to accept when energy increases. 

The results broadly replicate Ott's findings, as they show that participants treat high and low offers quite differently, with a further modulatory effect of energy, reflecting the energy specific parameters from Ott's model ($\beta_{basic}, \beta_{maxE}, \beta_{minE_LC}, \beta_{minE_HC}$).

In [11]:
#| label: fig-2
#| fig-cap: "Results of participants choice behaviour models. (A) Result of the model comparison, displaying the expected log pointwise predictive density (ELPD) 
#| associated with each model (closer to 0 indicate better fit). (B) Observed and fitted probability of accepting the offer separately for each offer by each of the models.
#| The single dots represent the average acceptance probability for each subject. (C) Fitted preference parameters of the preference model. Top left figure 
#| depicts the parameters associated with offer parameters, top right associated with each cost, bottom with each energy level. (D) Model fit for each participant
#| and model, color coded by model of best fit."
fig, ax = plt.subplot_mosaic("AB;CD", figsize=(8, 6), gridspec_kw=dict(height_ratios=[1, 1]))
# fig, ax = plt.subplots(2, 2, figsize=(12, 12))
cmap = matplotlib.colormaps.get_cmap('Set3')
# ================================================
# fig-2A: model comparison:
# elpd_loo bar plot with error bar
ax["A"].bar(    
    range(len(model_comparison)),
    model_comparison["elpd_loo"],
    yerr=model_comparison["se"],
    color=cmap([1, 2, 3])
)
# Decoration
ax["A"].set_xticks(range(len(model_comparison)))
ax["A"].set_xticklabels([
    "Preference model",
    "Hybrid model",
    "Planning model"], rotation=10
)
ax["A"].set_ylabel("ELPD (LOO)")
ax["A"].set_title("Model comparison")
ax["A"].spines[['right', 'top']].set_visible(False)
ax["A"].text(-0.1, 1.05, "A", transform=ax["A"].transAxes, 
              fontsize=16, fontweight='bold', va='top', ha='right'
              )

# ================================================
# fig-2B: Observed vs. fitted data of each model:
# Plot the true data against predicted values from each model:
pos = [1, 2, 3, 4, 6, 7, 8, 9, 11, 12, 13, 14, 16, 17, 18, 19]
cmap = matplotlib.colormaps.get_cmap('Set3')
ctr = 0
cols = ["response", "P(a=1)_preference", "P(a=1)_hybrid", "P(a=1)_planning"]
lbls = ["Observed", "Preference model", "Hybrid model", "Planning model"]
for o in [1, 2, 3, 4]:
    for col_i, col in enumerate(cols):
        # Group the data per participants:
        val = beh_data[beh_data['reward'] == o].groupby(['vpn'])[col].mean().reset_index()[col].to_numpy() 
        # Plot average across participants
        if o == 1:
            label = lbls[col_i]
        else:
            label = None
        ax["B"].bar(pos[ctr], np.mean(val), 
                    width=1, 
                    color=cmap(col_i), 
                    label=label)
        # Add error bars:
        ax["B"].errorbar(pos[ctr], np.mean(val), yerr=np.std(val), color='k', capsize=5)
        # Plot single participants values:
        ax["B"].scatter(np.random.uniform(0.2, 0.8, len(val)) + pos[ctr]-1/2, val, color='k', alpha=0.3)
        ctr += 1
ax["B"].set_xticks([2, 6, 10, 14])
ax["B"].set_xticklabels([1, 2, 3, 4])
ax["B"].set_xlabel("Offers")
ax["B"].set_ylabel("P(a=1)")
ax["B"].spines[['right', 'top']].set_visible(False)
ax["B"].text(-0.1, 1.05, "B", transform=ax["B"].transAxes, 
              fontsize=16, fontweight='bold', va='top', ha='right'
              )
ax["B"].legend(frameon=False);

# ================================================
# fig-2C: Plot the preferences parameters of the preference model:
# Split into 4 subplots:
fig.delaxes(ax["C"])

# Create a 2x2 subgrid in its place
subgs = ax["C"].get_subplotspec().subgridspec(2, 2)
ylim = []
# Top row: two separate axes
ax_offer = fig.add_subplot(subgs[0, 0])
ax_costs = fig.add_subplot(subgs[0, 1])
ax_energy = fig.add_subplot(subgs[1, :])

# Create new axes
ax_offer.text(-0.2, 1.15, "C", transform=ax_offer.transAxes, 
                fontsize=16, fontweight='bold', va='top', ha='right'
                ) 

# Plot offer parameters:
data = []
for var in ["B_is_1", "B_is_2", "B_is_3", "B_is_4"]:
    vals = traces['preferences_model'].posterior["beta_pref"].sel(coef_pref=var).values.flatten()
    data.append(vals)
ax_offer.violinplot(data, positions=range(len(data)), widths=0.8,
                       showmeans=False, showmedians=True,
                       showextrema=False, side="high");
ax_offer.set_xticks(range(len(data)))
ax_offer.set_xticklabels([1, 2, 3, 4])
ax_offer.hlines(0, -0.3, 3.5, color='gray', linestyles=':')
ax_offer.set_xlim([-0.3, 3.5])
ax_offer.spines[['right', 'top']].set_visible(False)
ax_offer.set_title("Offer")
ylim.append(ax_offer.get_ylim())

# Plot Costs parameters:
data = []
for var in ["B_islc", "B_ishc", "B_istrans1", "B_istrans2"]:
    vals = traces['preferences_model'].posterior["beta_pref"].sel(coef_pref=var).values.flatten()
    data.append(vals)
ax_costs.violinplot(data, positions=range(len(data)), widths=0.8,
                       showmeans=False, showmedians=True,
                       showextrema=False, side="high");
ax_costs.set_xticks(range(len(data)))
ax_costs.set_xticklabels(['cc=1', 'cc=2', 'fc=1', 'fc=2'], rotation=10)
ax_costs.hlines(0, -0.3, 3.5, color='gray', linestyles=':')
ax_costs.set_xlim([-0.3, 3.5])
ax_costs.spines[['right', 'top']].set_visible(False)
ax_costs.set_title("Costs")
ylim.append(ax_costs.get_ylim())

# Plot energy parameters:
data = []
for var in ["B_e_is_0", "B_e_is_1", "B_e_is_2", "B_e_is_3", 
            "B_e_is_4", "B_e_is_5", "B_e_is_6"]:
    vals = traces['preferences_model'].posterior["beta_pref"].sel(coef_pref=var).values.flatten()
    data.append(vals)
ax_energy.violinplot(data, positions=range(len(data)), widths=0.8, 
                       showmeans=False, showmedians=True,
                       showextrema=False, side="high");
ax_energy.set_xticks(range(len(data)))
ax_energy.set_xticklabels([0, 1, 2, 3, 4, 5, 6])
ax_energy.hlines(0, -0.3, 6.5, color='gray', linestyles=':')
ax_energy.set_xlim([-0.3, 6.5])
ax_energy.set_xlabel('Energy')
ax_energy.spines[['right', 'top']].set_visible(False)
# ax_energy.set_title("Energy parameters")
ylim.append(ax_energy.get_ylim())

ax_costs.set_ylim([np.min(ylim), np.max(ylim)])
ax_offer.set_ylim([np.min(ylim), np.max(ylim)])
ax_energy.set_ylim([np.min(ylim), np.max(ylim)])

# ================================================
# Single participant model fit:
ctr_mdl = {mdl: 0 for mdl in loo_results.keys()}
cmap = ["k", "tab:orange", "tab:green"]
for sub in loos_sub['vpn']:
    # Extract the data from this participant:
    sub_loo = loos_sub[loos_sub["vpn"] == sub][loo_results.keys()]
    # Extract the winning model:
    winning = sub_loo.idxmax(axis=1).values[0]
    if ctr_mdl[winning] == 0:
        lbl = winning
    else:
        lbl = None
    # Determine the color accordingly:
    c = cmap[list(loo_results.keys()).index(winning)]
    if winning == "planning_preferences_interaction_model":
        alpha = 0.5
        zorder = 0
    else:
        alpha = 1
        zorder = 1000
    ax["D"].scatter([1, 2, 3], np.squeeze(sub_loo.to_numpy()), color=c, alpha=alpha, zorder=zorder)
    ax["D"].plot([1, 2, 3], np.squeeze(sub_loo.to_numpy()), color=c, label=lbl, alpha=alpha, zorder=zorder)
    ctr_mdl[winning] += 1
ax["D"].set_xticks([1, 2, 3])
ax["D"].set_xticklabels(loo_results.keys())
ax["D"].set_ylabel("Sum of LOO values")
ax["D"].spines[['right', 'top']].set_visible(False)
ax["D"].legend(frameon=False)
ax["D"].text(-0.1, 1.05, "D", transform=ax["D"].transAxes, 
              fontsize=16, fontweight='bold', va='top', ha='right'
              )
plt.tight_layout();


<Figure size 2400x1800 with 6 Axes>

In [12]:
# Calculate the interaction confidence intervals:
mu_interaction = traces['preferences_model'].posterior["beta_interaction"].values.flatten().mean()
ci_interaction = az.hdi(traces['preferences_model'].posterior["beta_interaction"].values)[0]

Importantly, our model revealed a positive interaction between preferences entropy and decision values ($\beta=$`{python} f"{mu_interaction:.2f}"`, CI=[`{python} f"{ci_interaction[0]:.2f}, {ci_interaction[1]:.2f}"`], see @fig-3 A), indicating that participants weight decision values higher when their preferences are weak and vice versa. @fig-3 B depicts participants responses as a function of decision values and fitted preferences, the grey dots represent trials in which both the preferences and decision values were matched (either both positive or both negative), while colored markers indicate trials in which preferences and decision values were misaligned, color coded by participants responses (green triangles: accept, yellow hexagon: reject). Importantly, rejected trials in the upper left quadrant of the figure indicate that participans responses follows the decision values but goes against the preference, while accepting indicates that participants actions follows the preference to the detriment of the decision values, and vice versa in the bottom right quadrant. 

As we can see from @fig-3 B, when preferences are positive and decision values negative (top left quadrant), participants tend to accept the offer when the decision values are close to zero, reflected in the green triangles data points clustering towards 0 on the x axis. In contrast, participants tend to reject the reward more often when the decision values grow more negative, illustrated by the increased number of yellow dots on the left extremities of the top left quadrant. Similarly, when decision values are positive and preferences negative (bottom right quadrant), participants tend to reject the offer more often when the decision values are small, and accept the offer more often when the decision values grow larger (though the number of samples is much lower in this quadrant). These observations reflect the positive interaction between preferences entropy and decision values, indicating that participants rely more on preferences when decision values are close to 0 and vice versa. 

In [13]:
#| label: fig-3
#| fig-cap: "Interaction between preferences and decision values. (A) Posterior probability of the interaction parameter of the preference model
#| (B) Shows participants responses as a function of decision values and preference scores. On the off diagonal quadrant (top left and bottom right),
#| the dot color indicate whether participants accepted the reward (green triangles) or rejected the reward (yellow octogons). The number in the legend 
#| indicate hte number of trials of the corresponding colors."
fig, ax = plt.subplot_mosaic("AB", figsize=(8, 6), gridspec_kw=dict(width_ratios=[0.25, 1]))
ax["A"].text(-0.1, 1.05, "A", transform=ax["A"].transAxes, 
              fontsize=16, fontweight='bold', va='top', ha='right'
              )
# Plot the interaction estimated parameter:
vals = traces['preferences_model'].posterior["beta_interaction"].values.flatten()

ax["A"].violinplot([vals], positions=[1], 
                    showmeans=False,
                    showmedians=True,
                    showextrema=False,
                    side="high");
ax["A"].set_xticks([])
ax["A"].set_xlabel("Interaction")
ax["A"].spines[['right', 'top']].set_visible(False)
ax["A"].set_ylabel("Preference parameters")
ax["A"].hlines(0, 0.8, 2.3, color='gray', linestyles=':')
ax["A"].set_xlim([0.8, 1.3])

cmap = matplotlib.colormaps.get_cmap('Set3')
# Add the preferences as an additional column to the data:
beh_data['preference_score'] = np.mean(traces['preferences_model'].posterior["preference"], axis=(0, 1)).to_numpy()
beh_data['response_centered'] = beh_data['response'] - 0.5

# mismatch 1:
mismatch_accept = beh_data[(np.sign(beh_data['dv']) != np.sign(beh_data['preference_score'])) & (beh_data['response_centered'] > 0)]
# mismatch 2:
mismatch_reject= beh_data[(np.sign(beh_data['dv']) != np.sign(beh_data['preference_score'])) & (beh_data['response_centered'] < 0)]

# Correct responses in the absence of mismatchs:
correct_responses = beh_data[(np.sign(beh_data['dv']) == np.sign(beh_data['preference_score']))]

# Plot each mismatch
ax["B"].scatter(correct_responses['dv'], correct_responses['preference_score'], 
                color="grey", marker='o', alpha=0.1)
ax["B"].scatter(mismatch_reject['dv'], mismatch_reject['preference_score'], 
                color=cmap(1), edgecolors='black', linewidths=0.5, s=50,
                label=f'Response=Reject (N={mismatch_reject.shape[0]})', marker='h', alpha=0.8)
ax["B"].scatter(mismatch_accept['dv'], mismatch_accept['preference_score'], 
                color=cmap(0), edgecolors='black', linewidths=0.5, s=50,
                label=f'Response=Accept (N={mismatch_accept.shape[0]})', marker='v', alpha=0.8)

# Decoration:
ax["B"].set_xlabel("Decision values")
ax["B"].set_ylabel("Preference score")
ax["B"].text(-0.05, 1.05, "B", transform=ax["B"].transAxes, 
              fontsize=16, fontweight='bold', va='top', ha='right'
              )
ax["B"].spines[['right', 'top']].set_visible(False)
plt.legend(frameon=False)
plt.tight_layout();

<Figure size 2400x1800 with 2 Axes>

### Reaction times
In a task as large as ours (448 single states), computing the exact value associated with each state is costly. In comparison, preferences, are presumably readily available to the participants, or requires minimal computation. The interaction between the two observed at the responses levels suggests that participant control the planning demands depending on the strength of their preferences, such that if their preferences are really strong in a particular state, they would devolve less effort to forward planning. Indeed, @ott2022forward observed that RT grew significantly more rapidly in intermediate offers as a function of the magnitude of decision values compared to extreme offers, indicating that participants dedicate less cognitive effort to the extreme offers, which they interpret as evidence that these constitute two distinct contexts.

Importantly, these differences in RT patterns observed between intermediate and extreme offers might in fact be driven by differences in overall preference strength between both these conditions rather than these conditions being treated as separate contexts. In other words, we hypothesized that participants investement in forward planning depends on the strength of their preferences, and we therefore predicted that reaction time (indexing forward planning) should reflect the strength of the decision values and of their preferences as well as the interaction between the two rather than an interaction between decision values and intermediate offers (as was found in @ott2022forward). Specifically, we expect a positive interaction between conflict (i.e. negative amplitude of the decision values) and the preference entropy, indicating that participants reaction time increase more rapidly as a function of decision values when preferences are weak (i.e. entropy is high). 

In line with this prediction, modelling reaction time as a function of preferences and conflict as well as the interaction between the two performs better than @ott2022forward original model (see @fig-4 A). We observed a positive interaction effect between preferences entropy and conflict (see @fig-4 B), indicating that participants reaction time is more strongly dependent on decision values when preferences are weak (i.e. high preference entropy, see @fig-4 C). Combined with the response choice models, these results reveal that the participants decreased accuracy and reaction time in extreme offers compared to intermediate offers reflects participants stronger preferences in these conditions. 

In [14]:
beh_data['logRT'] = np.log(beh_data['reaction_time'])

# Repeat Ott's RT model:
beh_data['conflict'] = -np.abs(beh_data['dv'])
beh_data['intermediate'] = beh_data['is_2'] + beh_data['is_3']
beh_data['entropy'] = np.mean(traces['preferences_model'].posterior["entropy"], axis=(0, 1))

traces_RT = {}

# ===================================================================
# Ott's RT model:
ott_rt_mdl = bmb.Model(
    "logRT ~ conflict + intermediate + conflict:intermediate +"
    "(conflict + intermediate|vpn)", 
    beh_data
    )
traces_RT['ott_rt_mdl'] = ott_rt_mdl.fit(
    draws=1000, target_accept=0.85, idata_kwargs={"log_likelihood": True},
)

# ===================================================================
# Conflict entropy interaction model:
pref_conflict_rt_model = bmb.Model(
    "logRT ~ conflict + entropy + conflict:entropy +"
    "(conflict + entropy|vpn)", 
    beh_data
    )
traces_RT['preference_conflict_rt_model'] = pref_conflict_rt_model.fit(
    draws=1000, target_accept=0.85, idata_kwargs={"log_likelihood": True},
)

# Compare the models:
rt_model_comparison = az.compare(traces_RT)

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [sigma, Intercept, conflict, intermediate, conflict:intermediate, 1|vpn_sigma, 1|vpn_offset, conflict|vpn_sigma, conflict|vpn_offset, intermediate|vpn_sigma, intermediate|vpn_offset]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 43 seconds.


Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [sigma, Intercept, conflict, entropy, conflict:entropy, 1|vpn_sigma, 1|vpn_offset, conflict|vpn_sigma, conflict|vpn_offset, entropy|vpn_sigma, entropy|vpn_offset]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 45 seconds.


In [15]:
#| label: fig-4
#| fig-cap: "Results of reaction time models. (A) Comparison of the model of RT as a function of decision values and intermediate offer against the model of RT
#| as a function of decision values and preferences. (B) Posterior distribution of the fitted parameter. (C) Posterior predictive curve depicting the predicted
#| reaction time as a function of the conflict values, separately for preference entropy of 0, 0.5 and 1.0"
fig, ax = plt.subplot_mosaic("AB", figsize=(8, 6), gridspec_kw=dict(width_ratios=[0.5, 1]))
# ================================================
# fig-2A: model comparison:
# elpd_loo bar plot with error bar
ax["A"].bar(    
    range(len(rt_model_comparison)),
    rt_model_comparison["elpd_loo"],
    yerr=rt_model_comparison["se"],
    color=cmap([1, 2])
)
# Decoration
ax["A"].set_xticks(range(len(rt_model_comparison)))
ax["A"].set_xticklabels([
    "DV x Preference",
    "DV x Intermediate"], rotation=10)
ax["A"].set_ylabel("ELPD (LOO)")
ax["A"].set_title("Model comparison")
ax["A"].spines[['right', 'top']].set_visible(False)
ax["A"].text(-0.1, 1.05, "A", transform=ax["A"].transAxes, 
              fontsize=16, fontweight='bold', va='top', ha='right'
              )

# ================================================
# fig-B: Plot the preferences parameters of the preference & conflict model:
# Split into 4 subplots:
fig.delaxes(ax["B"])

# Create a 2x2 subgrid in its place
subgs = ax["B"].get_subplotspec().subgridspec(2, 1)
ylim = []
# Top row: two separate axes
ax_param = fig.add_subplot(subgs[0])
ax_rt_curves = fig.add_subplot(subgs[1])

# Create new axes
ax_param.text(-0.1, 1.1, "B", transform=ax_param.transAxes, 
                fontsize=16, fontweight='bold', va='top', ha='right'
                ) 
ax_rt_curves.text(-0.1, 1.05, "C", transform=ax_rt_curves.transAxes, 
                fontsize=16, fontweight='bold', va='top', ha='right'
                ) 
params = [
    "conflict", "entropy", "conflict:entropy"
]
x_labels = ["conflict", "entropy", "conflict:entropy"]

ax_param.set_ylabel("Estimated parameters")
data = []
for var in params:
    vals = traces_RT['preference_conflict_rt_model'].posterior[var].values.flatten()
    data.append(vals)
ax_param.violinplot(data, positions=range(len(params)), 
                    showmeans=False,
                    showmedians=True,
                    showextrema=False,
                    side="high");
ax_param.set_xticks(range(len(params)))
ax_param.set_xticklabels(x_labels)
ax_param.spines[['right', 'top']].set_visible(False)
ax_param.set_title("Parameters")
ax_param.hlines(0, -0.2, 2.3, color='gray', linestyles=':')
ax_param.set_xlim([-0.2, 2.3])

# ================================================
# fig-C: Plot the preferences parameters of the preference & conflict model:
pred_df = bmb.interpret.predictions(
    pref_conflict_rt_model,
    traces_RT['preference_conflict_rt_model'],
    conditional={
        "conflict": np.linspace(beh_data['conflict'].min(), beh_data['conflict'].max(), 100),
        "entropy": [0, 0.5, 1]
    },
)

pred_df["RT"] = np.exp(pred_df["estimate"])
pred_df["RT_low"] = np.exp(pred_df["lower_3.0%"])
pred_df["RT_up"] = np.exp(pred_df["upper_97.0%"])

for ent in pred_df["entropy"].unique():
    subset = pred_df[pred_df["entropy"] == ent]
    ax_rt_curves.plot(subset["conflict"], subset["RT"], label=f"entropy={ent}")
    ax_rt_curves.fill_between(subset["conflict"], subset["RT_low"], subset["RT_up"], alpha=0.5)
ax_rt_curves.spines[['right', 'top']].set_visible(False)
ax_rt_curves.set_xlabel('Conflict')
ax_rt_curves.set_ylabel('Reaction time (s.)')
ax_rt_curves.legend(frameon=False)
plt.tight_layout();


Default computed for unspecified variable: vpn


<Figure size 2400x1800 with 3 Axes>

### Variables weighting

We have observed that both response selection and reaction time reflects value based combined with preferences and that these preferences themselves reflect the structure of the task. Specifically, our results suggest that participants have preferences related to each factors of the task that then get combined to generate a composite preference score in a given state. Importantly, not  all features of the task are weighted equally. Indeed, we can see in @fig-2 C that the parameters associated with offers and energy weight higher than cost in the preference. To further characterize these differences, we used variance partitioning analysis within each subject to quantify the amount of variance accounted for by each experimental conditions. 

@fig-5 displays the difference in fit between the full preference model against models in which the preferences related to each experimental factors were removed, larger values indicate that the factor account for a more important proportion of the full model's fit (i.e. the preferences associated with this factor have a larger impact on participants choice). These results indicate that energy followed offer have a much larger impact on participants final responses, compared to costs. Why is that? It could be that humans are generally biased towards energy and rewards due to the prevalance of those dimensions in their environment. Alternatively, it could be that based on the structure of the task, following general heuristic associated with these two dimensions is efficient in yielding large return without relying on planning (which also assumes that the preferences participants have are also good for reward maximization, so double whammy). 

To investigate the latter possibility, we relied on posterior distribution sampling to evaluate the return that would be obtained if participants follow only the preferences related to each aspect of the task without relying on planning nor on any other preferences. 

In [16]:
# Fit the model without the offer related preferences:
traces_reduced = {}
if os.path.exists("./data/bids/limited_energy/derivatives/models/preferences_model-v2.nc"):
    idata = az.from_netcdf("./data/bids/limited_energy/derivatives/models/preferences_model-v2.nc")
    traces_reduced['preferences_model-v2'] = idata
else:
    preference_columns = [
        'is_1', 'is_2', 'is_3', 'is_4', 
        'is_lc', 'is_hc', 
        'is_lfc', 'is_hfc',
        'e_is_0', 'e_is_1', 'e_is_2', 'e_is_3', 'e_is_4', 'e_is_5', 'e_is_6'
    ]
    idata = preference_model(beh_data['response'],  # Subjects responses
                                np.squeeze(beh_data[['dv']].to_numpy()),  # Optimal decision values
                                beh_data[preference_columns],  # Preferences regressors
                                subj_idx_raw, subj_labels)
    # Add the idata to the rest:
    traces_reduced['preferences_model-v2'] = idata
    # Save the trace to file:
    az.to_netcdf(traces_reduced['preferences_model-v2'], "./data/bids/limited_energy/derivatives/models/preferences_model-v2.nc")

if os.path.exists("./data/bids/limited_energy/derivatives/models/preferences_model_no_offer_trace.nc"):
    idata = az.from_netcdf("./data/bids/limited_energy/derivatives/models/preferences_model_no_offer_trace.nc")
    traces_reduced['offer'] = idata
else:
    preference_columns = [
        'is_lc', 'is_hc', 
        'is_lfc', 'is_hfc',
        'e_is_0', 'e_is_1', 'e_is_2', 'e_is_3', 'e_is_4', 'e_is_5', 'e_is_6'
    ]
    idata = preference_model(beh_data['response'],  # Subjects responses
                                np.squeeze(beh_data[['dv']].to_numpy()),  # Optimal decision values
                                beh_data[preference_columns],  # Preferences regressors
                                subj_idx_raw, subj_labels)
    # Add the idata to the rest:
    traces_reduced['offer'] = idata
    # Save the trace to file:
    az.to_netcdf(traces_reduced['offer'], "./data/bids/limited_energy/derivatives/models/preferences_model_no_offer_trace.nc")

# Fit the model without the offer related preferences:
if os.path.exists("./data/bids/limited_energy/derivatives/models/preferences_model_no_costs_trace.nc"):
    idata = az.from_netcdf("./data/bids/limited_energy/derivatives/models/preferences_model_no_costs_trace.nc")
    traces_reduced['costs'] = idata
else:
    # Fit the model without the costs related preferences:
    preference_columns = [
        'is_1', 'is_2', 'is_3', 'is_4', 
        'is_lfc', 'is_hfc',
        'e_is_0', 'e_is_1', 'e_is_2', 'e_is_3', 'e_is_4', 'e_is_5', 'e_is_6'
    ]
    idata = preference_model(beh_data['response'],  # Subjects responses
                                np.squeeze(beh_data[['dv']].to_numpy()),  # Optimal decision values
                                beh_data[preference_columns],  # Preferences regressors
                                subj_idx_raw, subj_labels)
    # Add the idata to the rest:
    traces_reduced['costs'] = idata
    # Save the trace to file:
    az.to_netcdf(traces_reduced['costs'], "./data/bids/limited_energy/derivatives/models/preferences_model_no_costs_trace.nc")

# Fit the model without the offer related preferences:
if os.path.exists("./data/bids/limited_energy/derivatives/models/preferences_model_no_transition_trace.nc"):
    idata = az.from_netcdf("./data/bids/limited_energy/derivatives/models/preferences_model_no_transition_trace.nc")
    traces_reduced['future_cost'] = idata
else:
    # Fit the model without the transitions related preferences:
    preference_columns = [
        'is_1', 'is_2', 'is_3', 'is_4', 
        'is_lc', 'is_hc', 
        'e_is_0', 'e_is_1', 'e_is_2', 'e_is_3', 'e_is_4', 'e_is_5', 'e_is_6'
    ]
    idata = preference_model(beh_data['response'],  # Subjects responses
                                np.squeeze(beh_data[['dv']].to_numpy()),  # Optimal decision values
                                beh_data[preference_columns],  # Preferences regressors
                                subj_idx_raw, subj_labels)
    # Add the idata to the rest:
    traces_reduced['future_cost'] = idata
    # Save the trace to file:
    az.to_netcdf(traces_reduced['future_cost'], "./data/bids/limited_energy/derivatives/models/preferences_model_no_transition_trace.nc")

# Fit the model without the offer related preferences:
if os.path.exists("./data/bids/limited_energy/derivatives/models/preferences_model_no_energy_trace.nc"):
    idata = az.from_netcdf("./data/bids/limited_energy/derivatives/models/preferences_model_no_energy_trace.nc")
    traces_reduced['energy'] = idata
else:
    # Fit the model without the energy related preferences:
    preference_columns = [
        'is_1', 'is_2', 'is_3', 'is_4', 
        'is_lc', 'is_hc', 
        'is_trans1', 'is_trans2', 'is_trans3', 'is_trans4', 
    ]
    idata = preference_model(beh_data['response'],  # Subjects responses
                                np.squeeze(beh_data[['dv']].to_numpy()),  # Optimal decision values
                                beh_data[preference_columns],  # Preferences regressors
                                subj_idx_raw, subj_labels)
    # Add the idata to the rest:
    traces_reduced['energy'] = idata
    # Save the trace to file:
    az.to_netcdf(traces_reduced['energy'], "./data/bids/limited_energy/derivatives/models/preferences_model_no_energy_trace.nc")

# Compare against the full model:
model_comparison = az.compare(traces_reduced)

In [17]:
#| label: fig-5
#| fig-cap: "Importance of each parameters in participants response and optimal solution. (A) Depicts the difference in fit
#| between the full preference model and models in which each of the experimental factors were removed from preference estimations. 
#| Values closer to 0 indicate that the factor does not lead to a strong improvement of the model fit (i.e. the factor weight less in
#| participants final decision). "
fig, ax = plt.subplot_mosaic("A", figsize=(8, 6))  # , gridspec_kw=dict(width_ratios=[0.5, 1]))
# fig, ax = plt.subplots(2, 2, figsize=(12, 12))
cmap = matplotlib.colormaps.get_cmap('Set3')
# ================================================
# fig-2A: model comparison:
# elpd_loo bar plot with error bar
ax["A"].bar(    
    range(len(model_comparison)-1),
    model_comparison.iloc[1:]["elpd_diff"],
    yerr=model_comparison.iloc[1:]["dse"],
    color=cmap([1, 2, 3, 4])
)
# Decoration
ax["A"].set_xticks(range(len(model_comparison)-1))
ax["A"].set_xticklabels(model_comparison.index[1:])
ax["A"].set_ylabel("Δ ELPD (LOO)")
ax["A"].set_title("Model comparison")
ax["A"].spines[['right', 'top']].set_visible(False)
ax["A"].text(-0.1, 1.05, "A", transform=ax["A"].transAxes, 
              fontsize=16, fontweight='bold', va='top', ha='right'
              );

<Figure size 2400x1800 with 1 Axes>



To test this prediction, we generated a 

## Discussion


A limitation of the present work is we assume that preferences are fixed, thought they are likely to be dynamically updated throughout the task. Future work should aim to characterize preferences update mechanisms, by dynamically updating both the reward and transitional probabilty structure of the task, either in a gradually or chunk-wise (i.e. having separate experimental blocks with different task structures).

:::{#refs}

:::